In [ ]:
%pip install --quiet --upgrade https://github.com/bayuokta/bi-tools/releases/download/v0.9.0/bitools_sync-0.9.0-py3-none-any.whl

# BI Tools sync database (Fabric notebook)

Menjalankan `bitools-sync`: katalog workspace, lineage, katalog report, snapshot
SQLite, lalu upload ke SharePoint.

**Peringatan:** cell kredensial di bawah memuat `CLIENT_SECRET` service principal.
Jangan aktifkan Git integration untuk workspace ini, batasi anggota workspace,
dan rotasi secret setiap 6 bulan.


In [ ]:
# Parameter notebook (aman diubah / dijalankan lewat pipeline)
MODE = "admin"            # auto | admin | member
FULL = True               # full scan lineage setiap run
INCLUDE_PERSONAL = False  # sertakan My Workspace (hanya mode admin)
UPLOAD = True             # unggah snapshot ke SharePoint

DATABASE = "/tmp/bi_tools_sync.db"
SNAPSHOT = "/tmp/bi_tools_snapshot.db"

SP_HOSTNAME = "astrainternational.sharepoint.com"
SP_SITE_PATH = "/teams/BIDataRepository"
SP_DRIVE_NAME = "Documents"
SP_FOLDER = "Apps/BI Tools"


## Kredensial service principal

Isi di sini (workspace tanpa Git integration). `TENANT_ID` dan `CLIENT_ID` bukan
rahasia; `CLIENT_SECRET` rahasia dan harus dirotasi berkala.


In [ ]:
TENANT_ID = "isi-tenant-id"
CLIENT_ID = "isi-client-id"
CLIENT_SECRET = "isi-client-secret"


In [ ]:
import sqlite3
import sys

import core.version
from sync.flow import SyncOptions, run_sync
from sync.graph import SharePointTarget

print("Python :", sys.version.split()[0])
print("Sync   :", core.version.__version__)
print("SQLite :", sqlite3.sqlite_version)


In [ ]:
from pathlib import Path

from core.auth import ServicePrincipal
from core.errors import ExtractionError

principal = ServicePrincipal.from_values(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
target = SharePointTarget.from_values(SP_HOSTNAME, SP_SITE_PATH, SP_DRIVE_NAME, SP_FOLDER)
options = SyncOptions(
    database=Path(DATABASE),
    snapshot=Path(SNAPSHOT),
    mode=MODE,
    full=FULL,
    include_personal=INCLUDE_PERSONAL,
    upload=UPLOAD,
    sharepoint=target,
)

try:
    result = run_sync(options, principal, log=print)
except ExtractionError as exc:
    print("SYNC GAGAL:", exc)
    raise

print("---")
print("Workspace  :", result.catalog.workspace_count if result.catalog else "-")
print("Lineage    :", f"{result.lineage.node_count} node / {result.lineage.edge_count} edge" if result.lineage else "-")
print("Report     :", result.report_count)
print("Snapshot   :", result.snapshot, result.size, "byte")
print("Upload     :", result.upload or "-")


In [ ]:
from core.snapshot import read_meta, sha256_file

meta = read_meta(Path(SNAPSHOT))
size = Path(SNAPSHOT).stat().st_size
checksum = sha256_file(Path(SNAPSHOT))
print("meta    :", meta)
print("size    :", size)
print("sha256  :", checksum)
if UPLOAD and result.upload:
    print("remote  :", result.upload.get("files", {}))


In [ ]:
# Bersihkan file sementara supaya run berikutnya mulai dari database baru.
for path in (DATABASE, SNAPSHOT, SNAPSHOT + ".sha256"):
    Path(path).unlink(missing_ok=True)
print("selesai")
